# code for topic modeling of episode annotations and recall transcripts

### imports

In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from os.path import join as opj
from hypertools.tools import format_data as fit_transform
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from num2words import num2words
from scipy.signal import resample
from scipy.interpolate import interp1d

# stops hypertools from opening subprocess
%matplotlib inline

In [2]:
# try to show progress bars for long-running cells
# to properly show progress bars, you'll need to install tqdm,
# as well as widgetsnbextension and ipywidgets to render the element
try:
    from tqdm import tqdm_notebook as tqdm
    tqdm_pbar = True
except ModuleNotFoundError:
    print('To enable progress bars, install tqdm module (`pip install tqdm`)')
    from IPython.display import clear_output
    tqdm_pbar = False

### set paths

In [3]:
data_dir = '../../data'
annot_dir = opj(data_dir, 'annotations')
transc_dir = opj(data_dir, 'transcriptions', 'manual')
ep_traj_dir = opj(data_dir, 'models', 'episodes', 'trajectories')
rec_traj_dir = opj(data_dir, 'models', 'recalls', 'trajectories')
pickle_dir = opj(data_dir, 'pickles')

### load data

In [4]:
# formatted annotations
annotations = {episode: pd.read_csv(opj(annot_dir, f'{episode}.csv')) 
               for episode in ['atlep1', 'atlep2', 'arrdev']}

# across-session subject IDs
id_maps = pd.read_pickle(opj(pickle_dir, 'id_maps.p'))

### topic modeling parameters

In [5]:
# text preprocessing
stop_words = stopwords.words('english')
extra_stopwords = ['like']
stop_words = stop_words.extend(extra_stopwords)

# combine some multiword phrases, subsitute expletives, etc.
substitutions = {
    # names
    'paper boy': 'paperboy',
    'earnest': 'earn',
    'vanessa': 'van',
    'george-michael': 'georgemichael',
    'flo rida': 'floxrida',
    'david': 'dave',
    # explatives
    'f-word': 'fuck',
    'n-word': 'nigga',
    'racial slur': 'nigga',
    # other words/phrases
    'low key': 'lowkey',
    'low-key': 'lowkey',
    'parking lot': 'parkinglot',
    'déja': 'deja',
    'ex girlfriend': 'exgirlfriend',
    'ex-girlfriend': 'exgirlfriend',
    'ex wife': 'exwife',
    'ex-wife': 'exwife',
    # surround potential sub-words with spaces
    ' cause ': ' because ',
    ' weed ': ' marijuana ',
    ' pot ': ' marijuana '
}

# convert treebank pos tags to wordnet pos tags
# consider pronouns as nouns, deteriminers as adjectives
# default to noun for all other tags
pos_mappings = defaultdict(lambda: 'n')
for tb_tag, wn_tag in zip(['N', 'P', 'V', 'J', 'D', 'R'],
                          ['n', 'n', 'v', 'a', 'a', 'r']):
    pos_mappings[tb_tag] = wn_tag


In [6]:
n_topics = 100
episode_wsize = 50    # annotations
recall_wsize = 200    # words

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

# timestamp of last video frame
# used for interpolating timeseries
endframe_times = {
    'atlep1': 1466.0,
    'atlep2': 1316.52,
    'arrdev': 1236.6
}

## functions

In [7]:
# lower_nopunc = [re.sub("[^\w\s-]+", '', chunk.lower()) for chunk in episode_bag]
# no_acc = [x.replace('é', 'e') for x in lower_nopunc]
# no_digit = [re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), chunk) for chunk in no_acc]
# spaced = [' '.join(x.replace(',', ' ').split()) for x in no_digit]

### for text preprocessing and document formatting

In [8]:
def format_text(text):
    text = ' '.join(list(text.dropna()))
    punc_stripped = re.sub("[^\w\s-]+", '', text.lower())
    no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), punc_stripped)
    spaced = ' '.join(no_digit.replace(',', ' ').split())
    return spaced

In [9]:
def lemmatize(text, pos_dict=pos_mappings):
    lemmatizer = WordNetLemmatizer()
    words_tags = pos_tag(text.split())
    lemmas = []
    for word, tag in words_tags:
        lemma = lemmatizer.lemmatize(word, pos_dict[tag[0]])
        lemmas.append(lemma)
    return ' '.join(lemmas)

In [10]:
def preprocess_text(data, data_type=None):
    if data_type == 'episode':
        df = data.loc[:, 'Narrative details (external events)':'Setting']
    elif data_type == 'recall':
        df = pd.DataFrame(np.atleast_2d(data))
    else:
        raise ValueError("Episode vs recall data not specified")
        
    words_bag = df.apply(lambda x: format_text(x), axis=1)
    # combine multiword tokens, standardize names, deal with euphemisms, etc.
    replaced = words_bag.replace(substitutions, regex=True)
    # remove remaining hyphens
    no_hyphen = replaced.replace('-', ' ', regex=True)
    lemmatized = no_hyphen.apply(lambda x: lemmatize(x))
    preprocessed = lemmatized.tolist()
    return preprocessed[0].split() if data_type == 'recall' else preprocessed

In [11]:
# atlep1testpath = opj(transc_dir, 'MD-021919-A-02', 'debugQzo2F:debugV7e7L', 'debugQzo2F:debugV7e7L-recall.txt')
# arrdevtestpath = opj(transc_dir, 'MD-021819-B-02', 'debugNf95q:debugeVCxY', 'debugNf95q:debugeVCxY-recall.txt')
# with open(atlep1testpath, 'r') as f:
#     a1_test = f.read()
# with open(arrdevtestpath, 'r') as f:
#     arrdev_test = f.read()

In [12]:
def create_windows(textlist, wsize, taper_beginning=False, taper_end=True):
    windows = []
    if taper_beginning:
        # first `wsize` windows start with first item and grow until 
        # full window size. Ensures first `wsize` items and last `wsize` 
        # items are in equal number of windows if taper_end is True
        for ix in range(1, wsize):
            windows.append(' '.join(textlist[0 : ix]))
    if taper_end:
        # continue appending windows through last item, though last 
        # `wsize` windows will contain < `wsize` items
        n_indices = len(textlist)
    else:
        # stop shifting sliding window when < `wsize` items remain
        n_indices = len(textlist) - (wsize - 1)
        
    for ix in range(n_indices):
        windows.append(' '.join(textlist[ix : ix + wsize]))
    return windows

### for interpolating episode trajectories

In [13]:
def find_midpoint_times(df, endframe_time):
    """
    returns list of timepoints at middle of each annotation segment
    """
    midpoint_times = []
    for i, tpt in enumerate(df['Onset time']):
        try:
            midpoint_times.append((tpt + df['Onset time'][i+1]) / 2)
        except KeyError:
            # use final frame's timestamp
            midpoint_times.append((tpt + endframe_time) / 2)
    return midpoint_times

In [14]:
def interpolate_trajectory(traj, new_xmax, documents, resolution=1):
    if isinstance(documents, pd.DataFrame):
        # get middle timepoint for each annotation
        curr_xvals = find_midpoint_times(documents, new_xmax)
    else:
        # scale number of recall windows to episode length
        curr_xvals = np.linspace(0, new_xmax, len(traj))
        
    new_timepoints = np.arange(int(round(new_xmax)), step=resolution)
    interp_func = interp1d(curr_xvals, traj, axis=0, fill_value='extrapolate')
    return interp_func(new_timepoints)

## main topic modeling function

In [15]:
def transform_text(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
                   corpus=None, interp_len=None, return_windows=False):
    if isinstance(documents, pd.DataFrame):
        data_type = 'episode'
        window_size = episode_wsize
    elif isinstance(documents, str):
        data_type = 'recall'
        window_size = recall_wsize
        if not corpus:
            raise ValueError("You must pass a training corpus to transform recall transcripts")
            
    processed_docs = preprocess_text(documents, data_type=data_type)
    windows = create_windows(processed_docs, window_size)
    corpus = windows if data_type == 'episode' and not corpus else corpus
    traj = fit_transform(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
    if interp_len:
        traj = interpolate_trajectory(traj, interp_len, documents)
    return (traj, windows) if return_windows else traj

In [16]:
# def fit_and_transform(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
#                         corpus=None, resample_shape=None, return_windows=False):
#     # handle annotations
#     if isinstance(documents, pd.DataFrame):
#         windows = create_windows(documents, data_type='episode')
#         corpus = windows if not corpus else corpus
#         # fit topic model and transform documents
#         traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
#         # interpolate to length of episode (seconds)
#         if return_windows:
#             return interpolate_episode(traj, documents), windows
#         else:
#             return interpolate_episode(traj, documents)
        
#     # handle recall transcripts
#     elif isinstance(documents, str):
#         if not corpus:
#             raise ValueError("You must pass a training corpus to transform recall transcripts")
#         windows = get_recall_windows(documents, recall_wsize)
#         traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
#         # resample to corresponding episode length
#         return resample(traj, resample_shape) 

## model episode content, get sliding windows for fitting recall models

In [21]:
episode_trajs = dict.fromkeys(annotations.keys())
recall_corpora = dict.fromkeys(annotations.keys())

for episode, annotations_df in annotations.items():
    interp_len = endframe_times[episode]
    traj, windows = transform_text(annotations_df, 
                                   interp_len=interp_len, 
                                   return_windows=True)
    episode_trajs[episode] = traj
    recall_corpora[episode] = windows

## save episode trajectories

In [ ]:
# for episode, traj in episode_trajs.items():
#     np.save(opj(ep_traj_dir, f'{episode}_trajectory'), traj)

## load in and model recall transcripts

In [ ]:
# # try to show progress bars for long-running cells
# # to properly show progress bars, you'll need to install tqdm,
# # as well as widgetsnbextension and ipywidgets to render the element
# try:
#     from tqdm import tqdm_notebook as tqdm
#     def iter_transcripts(top, **kwargs):
#         total = 0
#         for root, dirs, files in os.walk(top, **kwargs):
#             dirs[:] = [d for d in dirs if d != 'drops']
#             files[:] = [f for f in files if f.startswith('debug')]
#             for file in files:
#                 with open(opj(root, file), 'r') as f:
#                     total += len(f.read().split())

#         with tqdm(total=total, unit='transcripts', leave=False) as pbar:
#             for root, dirs, files in os.walk(top, **kwargs):
#                 dirs[:] = [d for d in dirs if d != 'drops']
#                 files[:] = [f for f in files if f.startswith('debug')]
#                 yield root, dirs, files
                
#                 for file in files:
#                     pass
    
# except ModuleNotFoundError:
#     print('To enable progress bars, install tqdm module (`pip install tqdm`)')
#     def walkdir()

In [71]:
# total_files = 0
# n_subjects = 57
# for root, dirs, files in os.walk(transc_dir):
#     dirs[:] = [d for d in dirs if d != 'drops']
#     files[:] = [f for f in files if f.startswith('debug')]
#     total_files += len(files)
#     for file in files:
#         # base progress on transcript length for cleaner estimate
#         with open(opj(root, file), 'r') as f:
#             total_files += len(f.read().split())

n_subjects = id_maps.shape[0]
total_files = n_subjects * 3

In [75]:
# if tqdm_pbar:
#     pbar = tqdm(total=total_files, unit='transcripts', leave=False)
# else:
#     pct, frac, curr = 0, 0, 0
#     bar, pad = '', ' '*80
#     pbar = f"{pct}% [{bar}{pad}] {frac} {curr}"

In [93]:
recall_trajectories = {rectype: {} for rectype in recall_corpora.keys()}

if tqdm_pbar:
    itersubjects = tqdm(id_maps.iterrows(), total=total_files, leave=False)
else:
    itersubjects = id_maps.iterrows()
        
for sid, turkids in itersubjects:
    for ses in [1, 2]:
        turkid = turkids[f'session {ses}']
        if ses == 1:
            rectypes = ['atlep1']
        elif 'A' in sid:
            rectypes = ['delayed', 'atlep2']
        else:
            rectypes = ['delayed', 'arrdev']
            
        for rectype in rectypes:
            if rectype == 'delayed':
                ext = 'delayed'
                corpus = recall_corpora['atlep1']
                interp_len = endframe_times['atlep1']
            else:
                ext = 'recall'
                corpus = recall_corpora[rectype]
                interp_len = endframe_times[rectype]
                
            fpath = opj(transc_dir, sid, turkid, f'{turkid}-{ext}.txt')
            try:
                with open(fpath, 'r') as f:
                    transcript = f.read()
                    
                traj = transform_text(transcript, corpus=corpus, interp_len=interp_len)
                recall_trajectories[rectype][turkid] = traj
                if tqdm_pbar:
                    itersubjects.update(1)
                    
            except FileNotFoundError:
                continue
            if not transcript:
                continue
                
            

In [162]:
# x = {'ep': episode_trajs, 'rec': recall_trajectories}
# with open('/Users/paxtonfitzpatrick/Desktop/tmptrajs.p', 'wb') as f:
#     pickle.dump(x, f)

In [17]:
with open('/Users/paxtonfitzpatrick/Desktop/tmptrajs.p', 'rb') as f:
    x = pickle.load(f)
episode_trajs = x['ep']
recall_trajectories = x['rec']

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
# fig, axarr = plt.subplots(nrows=7, ncols=5)
# fig.set_size_inches(25,20)
# axarr = axarr.flatten()
# ax_idx = 0
# for sid, row in id_maps.iterrows():
#     tid1 = row['session 1']
#     tid2 = row['session 2']
#     try:
#         immtraj = recall_trajectories['atlep1'][tid1]
#         deltraj = recall_trajectories['delayed'][tid2]
#         hyp.plot([immtraj, deltraj], ndims=2, reduce='ppca', ax=axarr[ax_idx], show=False)
#         ax_idx += 1
#     except KeyError:
#         pass

# axarr[ax_idx].axis('off')
# fig.subplots_adjust(hspace=.1, wspace=.1)
# # fig.savefig('/Users/paxtonfitzpatrick/Desktop/new_imm_del_umap.pdf')
# display(fig)

In [ ]:
recall_trajectories = {
    'atlep1' : [],
    'prediction' : [],
    'delayed' : [],
    'atlep2' : [],
    'arrdev' : []
}

total = sum([len([f for f in files if f.endswith('corrected.wav.txt')]) for r, d, files in os.walk(transc_dir)])
# walk transcription directory structure
currfile = 1
for root, dirs, files in os.walk(transc_dir):
    
    # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
    transcripts = [f for f in files if f.endswith('corrected.wav.txt')]
    for transc in transcripts:

        # assign correct episode windows, corresponding episode trajectory shape, and dict key
        if any('prediction' in t for t in transcripts) or 'delayed' in transc:
            corpus = atlep1_windows
            resample_shape = atlep1_traj.shape[0]
            if 'recall' in transc:
                rectype = 'atlep1'
            elif 'prediction' in transc:
                rectype = 'prediction'
            elif 'delayed' in transc:
                rectype = 'delayed'
            else:
                raise ValueError('Transcript is not a recognized option')
            
        elif '-A-' in root:
            corpus = atlep2_windows
            resample_shape = atlep2_traj.shape[0]
            rectype = 'atlep2'
            
        else:
            corpus = arrdev_windows
            resample_shape = arrdev_traj.shape[0]
            rectype = 'arrdev'
            
        with open(opj(root ,transc), 'r') as f:
            # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
            transcript = ' '.join([line.split(',')[0].lower() for line in f.read().split('\n')])
            
        # fit topic model to episode annotations, 
        print(f'modeling transcript {currfile}/{total}...    {transc}')
        p_traj = fit_and_transform(transcript, resample_shape=resample_shape, corpus=corpus)
        
        recall_trajectories[rectype].append((transc.split('-')[0],p_traj))
        currfile += 1

## save individual trajectories

In [ ]:
# for rectype, data in recall_trajectories.items():
#     for (turkid, traj) in data:
#         np.save(opj(rec_traj_dir, rectype, f'{turkid}.npy'), traj)

## create and save average recall trajectories

In [ ]:
# for rectype, data in recall_trajectories.items():
#     avg_trajectory = np.array([traj for (turkid, traj) in data]).mean(axis=0)
#     np.save(opj(rec_traj_dir, rectype, 'avg_trajectory.npy'), avg_trajectory)

In [150]:
# from IPython.display import clear_output

# for i in range(100):
#     print(i)
#     time.sleep(0.3)
#     clear_output(wait=True)

In [23]:
for i, j in zip((a1_test, a2_test, ad_test), [1466.0, 1316.52, 1236.6]):
    print(int(round(j)))
    print(i.shape[0] == int(round(j)))

1466
True
1317
True
1237
True
